# 02 — Data Prep, Filtering & Implicit-Feedback Conversion

Turns raw MovieLens-25M ratings into the inputs the recommender needs:
1. **Filter** consensus-bad movies (>=80% of ratings below 2.5).
2. **Convert to implicit feedback**: every observed rating is a positive; the
   rating value becomes a *confidence* weight `C_ui = 1 + alpha*r_ui` (Hu/Koren/Volinsky 2008).
3. **Leave-one-out split**: hold out each user's most-recent rating as the test item.
4. **Save artifacts** so later notebooks load instantly instead of re-reading 25M rows.

All shared logic lives in `recsys_utils.py` (imported as `ru`).

In [1]:
import os, sys
os.environ['OPENBLAS_NUM_THREADS'] = '1'   # implicit/ALS: BLAS threadpool hurts here
sys.path.insert(0, os.path.abspath('..'))   # project root, for recsys_utils
import numpy as np, pandas as pd, scipy.sparse as sp
import recsys_utils as ru
DATA, ART = '../data/ml-25m', '../artifacts'
os.makedirs(ART, exist_ok=True)
print('alpha=%g  bad_fraction=%g  min_user_ratings=%d'
      % (ru.ALPHA, ru.BAD_FRACTION, ru.MIN_USER_RATINGS))

alpha=40  bad_fraction=0.8  min_user_ratings=5


## 1. Load raw ratings

In [2]:
ratings = pd.read_csv(f'{DATA}/ratings.csv')
print(f'{len(ratings):,} ratings | {ratings.userId.nunique():,} users | '
      f'{ratings.movieId.nunique():,} movies')
ratings.head(3)

25,000,095 ratings | 162,541 users | 59,047 movies


,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828


## 2. Item filter — drop consensus-bad movies

A movie that almost everyone rates below 2.5 should never be recommended, so it only
adds noise and parameters. We drop those. **Polarized** movies (loved by some, hated by
others) are kept on purpose — learning *who* likes them is exactly what the model is for.

In [3]:
ratings, dropped = ru.filter_consensus_bad(ratings)
print(f'dropped {len(dropped):,} consensus-bad movies; '
      f'{ratings.movieId.nunique():,} movies remain')

dropped 3,634 consensus-bad movies; 55,413 movies remain


## 3. User filter + contiguous reindex

Drop users with fewer than 5 ratings (too little signal, and leave-one-out needs >=2),
then map raw ids to contiguous row/column indices for the sparse matrix.

In [4]:
uc = ratings.userId.value_counts()
ratings = ratings[ratings.userId.isin(uc[uc >= ru.MIN_USER_RATINGS].index)]
ratings, user_ids, movie_ids = ru.reindex(ratings)
n_users, n_items = int(ratings.u.max()+1), int(ratings.i.max()+1)
print(f'{len(ratings):,} ratings | {n_users:,} users | {n_items:,} items')

24,985,878 ratings | 162,541 users | 55,413 items


## 4. Implicit-feedback confidence

MovieLens gives *explicit* 1–5 ratings, but we model this as **implicit** top-N ranking.
Following Hu/Koren/Volinsky (2008):

- **preference** `p_ui = 1` for every observed rating (the user engaged with the movie),
- **confidence** `C_ui = 1 + alpha * r_ui` — a 5-star rating gives more confidence in that
  positive than a 3-star one.

We store the raw `r_ui` in the matrix; the `implicit` library multiplies by `alpha`, and the
from-scratch model (nb 04) applies `1 + alpha*r` explicitly.

## 5. Leave-one-out split

Each user's **most-recent** rating becomes the test item (predict the *next* interaction).
Test items whose movie never appears in training are item-cold-start cases — dropped here,
handled in notebook 06.

In [5]:
train, test = ru.leave_one_out_split(ratings)
print(f'{len(train):,} train rows | {len(test):,} test rows (one held-out item per user)')

24,823,337 train rows | 162,495 test rows (one held-out item per user)


In [6]:
train_mat = ru.build_confidence_matrix(train, n_users, n_items)
print('train matrix:', train_mat.shape, f'| {train_mat.nnz:,} nonzeros '
      f'| density {train_mat.nnz/(n_users*n_items):.5%}')

train matrix: (162541, 55413) | 24,823,337 nonzeros | density 0.27560%


## 6. Save artifacts

`train_mat.npz` (sparse confidence matrix), the held-out test arrays, and the id maps.

In [7]:
sp.save_npz(f'{ART}/train_mat.npz', train_mat)
np.savez(f'{ART}/test.npz', u=test.u.values, i=test.i.values)
np.savez(f'{ART}/maps.npz', user_ids=user_ids, movie_ids=movie_ids)
print('saved:', os.listdir(ART))

saved: ['train_mat.npz', 'maps.npz', 'test.npz']
